In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from termcolor import cprint
import results_analysis_utils as rutils
from IPython.display import display, Markdown
pd.set_option('display.max_columns', 100)

# EXECUTE COPY TO LOCAL DISK PRIMER!!!!!

%load_ext autoreload
%autoreload 2

# Configuración estética para artículos científicos (Paper-ready)
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
plt.rcParams['figure.dpi'] = 200 
plt.rcParams['savefig.dpi'] = 200


#===================================================================
# 1. PARAMS AND INITIALIZATIONS
# ===================================================================
# Ruta base donde están todas las carpetas de días
DATASET_ID ="DATASET_2"
BASE_DIR = Path("/home/slimbook/fish_sizing/ARTICLE/"+DATASET_ID)

# Carpeta SOLO para los resultados globales (se crea si no existe)
OUTPUT_DIR_GLOBAL = BASE_DIR / "aggregated_data"
os.makedirs(OUTPUT_DIR_GLOBAL, exist_ok=True)

ASPECT_RATIO_THR = 3.0
ANGLE_THR = 20.0
day = "2025_08_21"

global_single_dfs = []
global_multi_dfs = []

# ===================================================================
# 1. DICCIONARIOS DE GROUND TRUTH
# ===================================================================
peix_measures = {
    "lubina0": 31.5, "fish_1": 19.4, "caballa": 28.9, "fish_3": 21.6,
    "llobarro": 33.5, "ochoa_no_cinta": 30.5, "ochoa_cinta": 33.5, 
    "ochoa_petita": 30.5, "llobarro_vermell": 29.1, "error": -100 
}


reverse_peix_measures = {v: k for k, v in peix_measures.items()}

single_fish_gt = {"2025_08_21": 29.1}

multiple_fish_gt = {
    "2025_08_21": {
        "10-01-53_0_compressed-r201-end": {-1: "None", 1: "red", 3: "no_mark", 4: "no_mark", 10: "red", 
                                           11: "red", 13: "no_mark", 18: "red", 19: "no_mark", 20: "None", 
                                           31: "no_mark", 32: "red"},
        
        "10-07-59_0-r50-end": {-1: "None", 1: "no_mark", 2: "red", 3: "no_mark", 
                               5: "no_mark", 6: "red", 9: "no_mark", 12: "no_mark", 
                               13: "None", 14: "None", 15: "red", 16: "red", 
                               18: "red", 19: "red", 20: "red", 21: "no_mark", 
                               22: "None", 24: "red", 27: "no_mark", 28: "red"},
        
        
        "10-09-31_0-r0-344": {1: "no_mark", 4: "red", 5: "no_mark", 7: "no_mark", 8: "None", 
                              9: "red", 10: "no_mark", 12: "no_mark", 16: "red", 17: "no_mark"},

        "10-09-31_0-r398-end": {-1: "None", 1: "red", 2: "no_mark", 4: "no_mark", 
                                5: "no_mark", 7: "red", 8: "no_mark", 9: "no_mark", 
                                12: "red", 13: "red", 14: "red"},
        
        
        "10-11-02_0-r0-244": {-1: "None", 1: "no_mark", 2: "red", 
                              3: "red", 7: "no_mark", 9: "None", 14: "red"},
        

        "10-11-02_0-r295-end": {1: "red", 2: "no_mark", 4: "red", 
                                8: "None", 10: "red", 14: "no_mark"},
  
        
        "10-12-34_0-r215-end": {1: "red", 2: "red", 4: "no_mark", 
                                6: "red", 7: "None", 10: "red", 
                                12: "red", 14: "None", 15: "no_mark"},
                
        
        "10-14-05_0-r0-528": {-1: "None", 1: "red", 2: "no_mark", 6: "red", 
                              9: "no_mark", 10: "red", 12: "None", 13: "no_mark", 
                              15: "None", 16: "None", 17: "no_mark", 18: "red"},
        
        
        "10-15-37_0-r496-end": {1: "no_mark", 2: "red", 5: "red", 6: "no_mark"},
        
        "10-18-24_0_compressed-r47-end": {-1: "None", 1: "red", 5: "no_mark", 
                                          7: "no_mark", 10: "red", 11: "None", 
                                          12: "red", 17: "red", 18: "red"},
        
        "10-19-44_0_compressed-r383-534": {1: "no_mark", 2: "red"},
    
        
        "10-19-44_0_compressed-r588-end": {1: "red", 2: "no_mark"},

        "10-20-29_1_compressed-r0-78": {-1: "None", 1: "red", 
                                        2: "no_mark", 3: "no_mark", 4: "red"},
        
        "10-22-00_1_compressed": {-1: "None", 1: "red", 2: "no_mark",
                                  3: "red", 5: "None"},

        "10-37-15_1-r": {-1: "None", 1: "no_mark", 2: "red", 
                         3: "red", 6: "red", 8: "no_mark", 9: "error", 10: "error"},
        
        
        "13-05-55_0-r": {1: "red", 2: "no_mark", 3: "black", 4: "black", 5: "no_mark", 
                         8: "black", 10: "red", 11: "None", 15: "black", 20: "red", 
                         22: "None", 25: "None", 26: "no_mark", 27: "None", 28: "no_mark", 
                         30: "no_mark", 34: "no_mark", 38: "red", 39: "None", 42: "red", 
                         47: "no_mark", 49: "black"},
        
                
        "13-22-22_0_compressed-r60-297": {1: "no_mark", 2: "black", 3: "red", 6: "red", 
                                          7: "None", 9: "red", 11: "None", 13: "black"},
        

        "13-22-22_0_compressed-r378-463": {1: "black", 2: "no_mark", 3: "red", 4: "error"},
        
                
        "13-23-54_0_compressed-r0-304": {-1: "None", 2: "red", 3: "black", 5: "no_mark"},
        
        "13-23-54_0_compressed-r450-524": {1: "no_mark", 2: "black", 3: "red", 5: "None"},

        "13-23-54_0_compressed-r602-end": {1: "red", 2: "black", 3: "no_mark"}}
     
}

ALL_DAYS = list(set(list(single_fish_gt.keys()) + list(multiple_fish_gt.keys())))

# ===================================================================
# 3. BUCLE MAESTRO
# ===================================================================
cprint(f"🚀 INICIANDO PIPELINE DE AGREGACIÓN PARA {len(ALL_DAYS)} DÍAS", "white", "on_blue", attrs=["bold"])

for day_code in sorted(ALL_DAYS):
    cprint(f"\n" + "="*50, "cyan")
    cprint(f"📅 PROCESANDO DÍA: {day_code}", "cyan", attrs=["bold"])
    
    day_folder = BASE_DIR / day_code
    
    if not day_folder.exists():
        cprint(f"⚠️ La carpeta no existe. Busqué en: {day_folder}", "yellow")
        continue

    day_dfs = []

    # ---------------------------------------------------------------
    # A) PROCESAR SINGLE FISH
    # ---------------------------------------------------------------
    dir_single = day_folder / "single_fish"
    
    if dir_single.exists() and day_code in single_fish_gt:
        cprint(f"\n🐟 Buscando SINGLE FISH en {dir_single}", "magenta")
        df_single = rutils.aggregate_results_from_root_new(dir_single, day_code, output_csv_path=dir_single,results_foldername="results")
        
        if not df_single.empty:
            medida = single_fish_gt[day_code]
            especie = reverse_peix_measures.get(medida, "unknown_single_fish")
            
            df_single["gt"] = medida
            df_single["fish_id"] = especie
            df_single["abs_error_cm"] = abs((df_single["filtered_length"] * 100) - medida)
            df_single["failure_reason"] = np.nan
            df_single['unique_track'] = df_single['video_day'].astype(str) + "/" + df_single['video_name'].astype(str) + "/" + df_single['track_id'].astype(str)
            df_single["scenario"] = "single_fish"
            
            out_single = dir_single / f"{day_code}_single_fish_agg.csv"
            df_single.to_csv(out_single, index=False)
            
            global_single_dfs.append(df_single)
            day_dfs.append(df_single) # Añadimos al saco del día
            cprint(f"   ✅ Single Fish completado. Guardado en: {out_single}", "green")
    else:
        if not dir_single.exists():
            cprint(f"   ℹ️ No hay carpeta 'single_fish' para este día.", "dark_grey")
    
    # ---------------------------------------------------------------
    # B) PROCESAR MULTIPLE FISH
    # ---------------------------------------------------------------
    dir_multi = day_folder / "multiple_fish"  
    
    if dir_multi.exists() and day_code in multiple_fish_gt:
        cprint(f"\n🐠🐠 Buscando MULTIPLE FISH en {dir_multi}...", "magenta")
        df_multi_raw = rutils.aggregate_results_from_root_new(dir_multi, day_code, output_csv_path=dir_multi, results_foldername="results")
        
        # Make sure it is integer
        df_multi_raw['track_id'] = pd.to_numeric(df_multi_raw['track_id'], errors='coerce').fillna(-1).astype(int)
        
        if not df_multi_raw.empty:
            df_multi, huerfanos = rutils.inject_ground_truth(
                df_multi_raw, 
                {day_code: multiple_fish_gt[day_code]}, 
                peix_measures, 
                drop_unlabeled=False 
            )
            df_multi["scenario"] = "multiple_fish"
            
            out_multi = dir_multi / f"{day_code}_multiple_fish_agg.csv"
            df_multi.to_csv(out_multi, index=False)
            
            global_multi_dfs.append(df_multi)
            day_dfs.append(df_multi) # Añadimos al saco del día
            cprint(f"   ✅ Multiple Fish completado. Guardado en: {out_multi}", "green")
    else:
        if not dir_multi.exists():
             cprint(f"   ℹ️ No hay carpeta 'multiple_fish' para este día.", "dark_grey")

    # ---------------------------------------------------------------
    #  C) GUARDAR EL AGREGADO GLOBAL DEL DÍA EN LA RAÍZ
    # ---------------------------------------------------------------
    if day_dfs:
        df_day_all = pd.concat(day_dfs, ignore_index=True)
        out_day = day_folder / f"{day_code}_ALL_raw_aggregated.csv"
        df_day_all.to_csv(out_day, index=False)
        cprint(f"\n   📁 AGREGADO DEL DÍA CREADO: {out_day}", "cyan", attrs=["bold"])
        
        
# ===================================================================
# 4. CREAR LOS DATASETS MASIVOS GLOBALES
# ===================================================================
cprint("\n" + "="*50, "blue")
cprint(f"🌍 GENERANDO DATASETS GLOBALES EN: {OUTPUT_DIR_GLOBAL.name}", "white", "on_blue", attrs=["bold"])

df_master_single = pd.DataFrame()
df_master_multi = pd.DataFrame()

# 1. Guardar el global exclusivo de Single Fish
if global_single_dfs:
    df_master_single = pd.concat(global_single_dfs, ignore_index=True)
    out_single_global = OUTPUT_DIR_GLOBAL / f"{DATASET_ID}_ALL_single_fish_all_days.csv"
    df_master_single.to_csv(out_single_global, index=False)
    cprint(f"📦 Guardado: {out_single_global.name} ({len(df_master_single)} frames)", "cyan")

# 2. Guardar el global exclusivo de Multiple Fish
if global_multi_dfs:
    df_master_multi = pd.concat(global_multi_dfs, ignore_index=True)
    out_multi_global = OUTPUT_DIR_GLOBAL / f"{DATASET_ID}_ALL_multiple_fish_all_days.csv"
    df_master_multi.to_csv(out_multi_global, index=False)
    cprint(f"📦 Guardado: {out_multi_global.name} ({len(df_master_multi)} frames)", "cyan")

# 3. Guardar el SÚPER MEGA DATASET (Todos los días + Single + Multiple)
if not df_master_single.empty or not df_master_multi.empty:
    # Juntamos los dos mundos
    df_master_all = pd.concat([df_master_single, df_master_multi], ignore_index=True)
    
    # ⚡ Aplicamos tu clasificador vectorial (bordes, 3D, aspect ratio...) a toda la base de datos de golpe
    df_master_all = rutils.assign_failure_reasons(df_master_all, aspect_ratio_thr=ASPECT_RATIO_THR, angle_thr=ANGLE_THR)
    
    out_master = OUTPUT_DIR_GLOBAL / f"{DATASET_ID}_all_fish_all_days.csv"
    df_master_all.to_csv(out_master, index=False)
    cprint(f"🏆 DATASET MAESTRO GUARDADO: {out_master.name} ({len(df_master_all)} frames)", "green", attrs=["bold"])

🚀 INICIANDO PIPELINE DE AGREGACIÓN PARA 1 DÍAS

📅 PROCESANDO DÍA: 2025_08_21

🐟 Buscando SINGLE FISH en /home/slimbook/fish_sizing/ARTICLE/DATASET_2/2025_08_21/single_fish
🔍 Buscando datos RAW en: /home/slimbook/fish_sizing/ARTICLE/DATASET_2/2025_08_21/single_fish
📂 Encontrados 35 archivos de resultados crudos.

📊 TOTAL AGREGADO: 16380 líneas de datos.
   ✅ Single Fish completado. Guardado en: /home/slimbook/fish_sizing/ARTICLE/DATASET_2/2025_08_21/single_fish/2025_08_21_single_fish_agg.csv

🐠🐠 Buscando MULTIPLE FISH en /home/slimbook/fish_sizing/ARTICLE/DATASET_2/2025_08_21/multiple_fish...
🔍 Buscando datos RAW en: /home/slimbook/fish_sizing/ARTICLE/DATASET_2/2025_08_21/multiple_fish
📂 Encontrados 21 archivos de resultados crudos.

📊 TOTAL AGREGADO: 9831 líneas de datos.
✅ Ground Truth inyectado. Vídeos únicos: 21
   🐟 Tracks CON Ground Truth: 3
   ⚠️ Tracks SIN Ground Truth (NaN): 164
   ✅ Multiple Fish completado. Guardado en: /home/slimbook/fish_sizing/ARTICLE/DATASET_2/2025_08_21/